# KB Enrichment

Three goals:

1. Analyse the combined KB — sources, categories, lengths
2. Prepare a small labelled training set for the future classifier
3. Save the enriched KB with derived features

Input: `data/processed/kb_combined.csv`  
Outputs:
- `data/processed/kb_enriched.csv`
- `data/processed/classifier_train.csv`

In [1]:
import pandas as pd
from pathlib import Path

In [2]:
ROOT = Path.cwd().parent
PROCESSED_DIR = ROOT / "data" / "processed"

kb = pd.read_csv(PROCESSED_DIR / "kb_combined.csv")
print(kb.shape)
print(kb.columns.tolist())

(258494, 9)
['doc_id', 'source', 'source_id', 'category', 'question', 'context', 'answer', 'has_answer', 'has_context']


C:\Users\Matrix\AppData\Local\Temp\ipykernel_4428\4234780445.py:4: DtypeWarning: Columns (2,3) have mixed types. Specify dtype option on import or set low_memory=False.
  kb = pd.read_csv(PROCESSED_DIR / "kb_combined.csv")


In [3]:
kb.head(3)

,doc_id,source,source_id,category,question,context,answer,has_answer,has_context
0,0,medquad,0,susceptibility,Who is at risk for Lymphocytic Choriomeningiti...,NaN,LCMV infections can occur after exposure to fr...,True,False
1,1,medquad,1,symptoms,What are the symptoms of Lymphocytic Choriomen...,NaN,LCMV is most commonly recognized as causing ne...,True,False
2,2,medquad,2,susceptibility,Who is at risk for Lymphocytic Choriomeningiti...,NaN,Individuals of all ages who come into contact ...,True,False


## 1. Composition analysis

Which sources carry the KB, and which of them already have categories?

In [4]:
kb["source"].value_counts()

source
pubmedqa       211269
medredqa        29801
medquad         16407
medlineplus      1017
Name: count, dtype: int64

In [5]:
# Only MedQuAD has a populated category field. Everything else is blank.
kb["has_category"] = kb["category"].fillna("").astype(str).str.strip() != ""

kb.groupby("source")["has_category"].agg(["sum", "count", "mean"]).round(3)

,sum,count,mean
source,,,
medlineplus,0,1017,0.0
medquad,16407,16407,1.0
medredqa,0,29801,0.0
pubmedqa,0,211269,0.0


In [6]:
# What categories does MedQuAD actually provide?
kb.loc[kb["source"] == "medquad", "category"].value_counts()

category
information        4535
symptoms           2748
treatment          2442
inheritance        1446
frequency          1120
genetic changes    1087
causes              727
exams and tests     653
research            395
outlook             361
susceptibility      324
considerations      235
prevention          210
stages               77
complications        46
support groups        1
Name: count, dtype: int64

## 2. Length features

Word counts for question, context, and answer. Used later for chunking and to spot odd records.

In [7]:
for col in ["question", "context", "answer"]:
    kb[f"{col}_words"] = kb[col].fillna("").astype(str).str.split().str.len()

kb[["question_words", "context_words", "answer_words"]].describe().round(1)

,question_words,context_words,answer_words
count,258494.0,258494.0,258494.0
mean,14.1,185.7,52.5
std,5.0,91.8,83.1
min,1.0,0.0,0.0
25%,11.0,152.0,26.0
50%,14.0,192.0,37.0
75%,17.0,223.0,52.0
max,102.0,4518.0,4281.0


In [8]:
# Median length per source
kb.groupby("source")[["question_words", "context_words", "answer_words"]].median().round(0)

,question_words,context_words,answer_words
source,,,
medlineplus,4.0,0.0,192.0
medquad,7.0,0.0,138.0
medredqa,8.0,153.0,47.0
pubmedqa,15.0,198.0,34.0


## 3. Classifier training set

MedQuAD is the only source with real category labels. We map its native
`qtype` values onto the 5 categories we care about. MedQuAD does not have a
`Medication` label, so the classifier will be trained on 5 classes only.

Mapping used:

| MedQuAD qtype | Target category |
|---------------|-----------------|
| symptoms | Symptoms |
| exams and tests | Diagnosis |
| treatment | Treatment |
| susceptibility | Prevention |
| information, causes, outlook, considerations | General |

Genetic-specific qtypes (`inheritance`, `frequency`, `genetic changes`) and `research` are dropped — too specialised for a general consumer assistant.

In [9]:
QTYPE_MAP = {
    "symptoms": "Symptoms",
    "exams and tests": "Diagnosis",
    "stages": "Diagnosis",
    "treatment": "Treatment",
    "complications": "General",
    "susceptibility": "Prevention",
    "prevention": "Prevention",
    "information": "General",
    "causes": "General",
    "outlook": "General",
    "considerations": "General",
    "support groups": "General",
}

mq = kb[kb["source"] == "medquad"].copy()
mq["label"] = mq["category"].str.strip().map(QTYPE_MAP)
mq = mq.dropna(subset=["label"])

print("usable:", len(mq), "/", len(kb[kb["source"] == "medquad"]))
print(mq["label"].value_counts())

usable: 12359 / 16407
label
General       5905
Symptoms      2748
Treatment     2442
Diagnosis      730
Prevention     534
Name: count, dtype: int64


In [10]:
classifier_train = mq[["question", "label"]].reset_index(drop=True)
print(classifier_train.shape)
classifier_train.head(3)

(12359, 2)


,question,label
0,Who is at risk for Lymphocytic Choriomeningiti...,Prevention
1,What are the symptoms of Lymphocytic Choriomen...,Symptoms
2,Who is at risk for Lymphocytic Choriomeningiti...,Prevention


In [11]:
classifier_train.to_csv(PROCESSED_DIR / "classifier_train.csv", index=False)
print("saved")

saved


## 4. Save enriched KB

In [12]:
kb.to_csv(PROCESSED_DIR / "kb_enriched.csv", index=False)
print("kb_enriched.csv:", kb.shape)

kb_enriched.csv: (258494, 13)


Next: `04_eda.ipynb`